In [25]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [26]:
# --------------------------
# 1) Dataset Wine
# https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_wine.html
# https://archive.ics.uci.edu/dataset/109/wine
# --------------------------
data = load_wine()
X, y = data.data, data.target
mask = y != 2           # filtramos clase 2
X, y = X[mask], y[mask] # ahora binario (clases 0 y 1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Estandarización
scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train)
X_test  = scaler.transform(X_test)



In [27]:
print(X_train)

[[-0.23655888 -0.25179223 -0.19888842 ... -1.05993134 -1.1052945
  -0.89951312]
 [ 0.4712347  -1.14858802  0.08482006 ... -0.19522705 -2.17962267
  -0.12351168]
 [-1.20692103  2.58252536  0.19121074 ... -1.86287103  1.5695634
  -1.21939158]
 ...
 [-1.0585127  -0.5175095   0.29760142 ... -0.00993327 -0.60101801
  -1.01206295]
 [-0.32788708 -0.6835828  -1.12094097 ... -0.44228542 -2.92507487
  -1.01206295]
 [ 0.85937956 -0.1632198   0.08482006 ...  1.10182938 -0.11866659
   0.58732933]]


In [28]:
# --------------------------
# 2) Logistic Regression
# --------------------------
clf = LogisticRegression()
clf.fit(X_train, y_train)
yhat_tr_log = clf.predict(X_train)
yhat_te_log = clf.predict(X_test)
acc_tr_log = accuracy_score(y_train, yhat_tr_log) * 100
acc_te_log = accuracy_score(y_test,  yhat_te_log) * 100

print(f"LogisticRegression -> acc_train: {acc_tr_log:.2f}% | acc_test: {acc_te_log:.5f}%")

LogisticRegression -> acc_train: 100.00% | acc_test: 96.96970%


In [29]:
# --------------------------
# 3) Perceptrón (neurona con sigmoide + BCE, equivalente a una regresión logística)
# --------------------------

# 1. Preparo los datos
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor =torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

# 2. Defino el modelo
class Perceptron(nn.Module):
    def __init__(self, input_size, p_drop=0):
        super().__init__()
        self.dropout = nn.Dropout(p_drop)
        self.linear = nn.Linear(input_size, 1)
        self.activation = nn.Sigmoid()

    def forward(self, x):
        #x = self.dropout(x)
        x = self.linear(x)
        x = self.activation(x)
        return x

# 3. Inicializo el modelo
input_size = X_train_tensor.shape[1]
model = Perceptron(input_size)
criterion = nn.BCELoss() # Binary Cross-Entropy Loss for binary classification
optimizer = optim.SGD(model.parameters(), lr=0.1) # Stochastic Gradient Descent

# 4. Entreno el modelo
epochs = 1000
for epoch in range(epochs):
    # Forward pass
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)

    # Backpropagation
    optimizer.zero_grad()
    loss.backward() # Calculo los gradientes
    optimizer.step() # Modifico los pesos

    if (epoch + 1) % 100 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

# 5. Evaluo el modelo
with torch.no_grad():
    yhat_tr_perceptron = (model(X_train_tensor) > 0.5).float()
    acc_tr_log = (yhat_tr_perceptron == y_train_tensor).sum().item() / len(y_train_tensor) * 100
    yhat_te_perceptron = (model(X_test_tensor) > 0.5).float()
    acc_te_log = (yhat_te_perceptron == y_test_tensor).sum().item() / len(y_test_tensor) * 100
    print(f"Perceptron (pytorch) -> acc_train: {acc_tr_log:.2f}% | acc_test: {acc_te_log:.5f}%")
    accuracy_tensor = accuracy_score(y_train_tensor, yhat_tr_perceptron)
    print(accuracy_tensor)
    accuracy_tensor_test = accuracy_score(y_test_tensor, yhat_te_perceptron)
    print(accuracy_tensor_test)


Epoch [100/1000], Loss: 0.0865
Epoch [200/1000], Loss: 0.0535
Epoch [300/1000], Loss: 0.0399
Epoch [400/1000], Loss: 0.0322
Epoch [500/1000], Loss: 0.0272
Epoch [600/1000], Loss: 0.0237
Epoch [700/1000], Loss: 0.0210
Epoch [800/1000], Loss: 0.0189
Epoch [900/1000], Loss: 0.0172
Epoch [1000/1000], Loss: 0.0158
Perceptron (pytorch) -> acc_train: 100.00% | acc_test: 96.96970%
1.0
0.9696969696969697


Ejercicios:

1) Implemetar la clase MLP3 (perceptrón multicapa de 3 capas densas).
2) Entrenar, evaluar y comparar resutlados para MLP3.

In [30]:
# 2. Defino el modelo
class MLP3(nn.Module):
    def __init__(self, input_size, p_drop=0):
        super().__init__()
        self.dropout = nn.Dropout(p_drop)

        self.linear1 = nn.Linear(input_size, input_size)
        self.linear2 = nn.Linear(input_size, input_size)
        self.linear3 = nn.Linear(input_size, 1)

        self.activation = nn.ReLU()
        self.activation_output = nn.Sigmoid()


    def forward(self, x):
        #x = self.dropout(x)
        x = self.linear1(x)
        x = self.activation(x)

        x = self.dropout(x)
        x = self.linear2(x)
        x = self.activation(x)

        x = self.linear3(x)
        x = self.activation_output(x)
        return x

# 3. Inicializo el modelo
input_size = X_train_tensor.shape[1]
model_mlp3 = MLP3(input_size)
criterion = nn.BCELoss() # Binary Cross-Entropy Loss for binary classification
optimizer = optim.SGD(model_mlp3.parameters(), lr=0.1) # Stochastic Gradient Descent

# 4. Entreno el modelo
epochs = 1000
for epoch in range(epochs):
    # Forward pass
    outputs = model_mlp3(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)

    # Backpropagation
    optimizer.zero_grad()
    loss.backward() # Calculo los gradientes
    optimizer.step() # Modifico los pesos

    if (epoch + 1) % 100 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

Epoch [100/1000], Loss: 0.0556
Epoch [200/1000], Loss: 0.0108
Epoch [300/1000], Loss: 0.0049
Epoch [400/1000], Loss: 0.0029
Epoch [500/1000], Loss: 0.0020
Epoch [600/1000], Loss: 0.0015
Epoch [700/1000], Loss: 0.0012
Epoch [800/1000], Loss: 0.0010
Epoch [900/1000], Loss: 0.0009
Epoch [1000/1000], Loss: 0.0007


In [31]:
# 5. Evaluo el modelo
with torch.no_grad():
    yhat_tr_perceptron_mlp3 = (model_mlp3(X_train_tensor) > 0.5).float()
    acc_tr_log_mlp3 = accuracy_score(y_train_tensor, yhat_tr_perceptron_mlp3)
    yhat_te_perceptron_mlp3 = (model_mlp3(X_test_tensor) > 0.5).float()
    acc_te_log_mlp3 = accuracy_score(y_test_tensor, yhat_te_perceptron_mlp3)
    print(f"MLP3 (pytorch) -> acc_train: {acc_tr_log_mlp3*100:.2f}% | acc_test: {acc_te_log_mlp3*100:.5f}%")

MLP3 (pytorch) -> acc_train: 100.00% | acc_test: 96.96970%


Ahora quiero hacer la clasificacion con las 3 clases

In [32]:
X_full, y_full = data.data, data.target

X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(
    X_full, y_full, test_size=0.25, random_state=42, stratify=y_full
)

# Estandarización
scaler = StandardScaler().fit(X_train_full)
X_train_full = scaler.transform(X_train_full)
X_test_full  = scaler.transform(X_test_full)


In [ ]:
# --------------------------
# 4) MLP3 multiclase (3 clases)
# --------------------------

# 1. Preparo los datos
X_train_tensor_full = torch.tensor(X_train_full, dtype=torch.float32)
X_test_tensor_full = torch.tensor(X_test_full, dtype=torch.float32)

y_train_tensor_full = torch.tensor(y_train_full, dtype=torch.long).reshape(-1)
y_test_tensor_full = torch.tensor(y_test_full, dtype=torch.long).reshape(-1)

# 2. Defino el modelo
class MLP3_Classifier(nn.Module):
    def __init__(self, input_size, p_drop=0):
        super().__init__()
        self.dropout = nn.Dropout(p_drop)
        self.linear1 = nn.Linear(input_size, input_size)
        self.linear2 = nn.Linear(input_size, input_size)
        self.linear3 = nn.Linear(input_size, 3)

        self.activation = nn.ReLU()

    def forward(self, x):
        #x = self.dropout(x)
        x = self.linear1(x)
        x = self.activation(x)

        x = self.linear2(x)
        x = self.activation(x)

        x = self.linear3(x)
        return x

# 3. Inicializo el modelo
input_size = X_train_tensor_full.shape[1]
model_classifier = MLP3_Classifier(input_size)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model_classifier.parameters(), lr=0.1) # Stochastic Gradient Descent

# 4. Entreno el modelo
epochs = 1000
for epoch in range(epochs):
    # Forward pass
    outputs = model_classifier(X_train_tensor_full)
    loss = criterion(outputs, y_train_tensor_full)

    # Backpropagation
    optimizer.zero_grad()
    loss.backward() # Calculo los gradientes
    optimizer.step() # Modifico los pesos

    if (epoch + 1) % 100 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

# 5. Evaluo el modelo
model_classifier.eval()

with torch.no_grad():
    logits_train = model_classifier(X_train_tensor_full)
    pred_train = torch.argmax(logits_train, dim=1)

    logits_test = model_classifier(X_test_tensor_full)
    pred_test = torch.argmax(logits_test, dim=1)

    acc_train = (pred_train == y_train_tensor_full).float().mean().item() * 100
    acc_test = (pred_test == y_test_tensor_full).float().mean().item() * 100

    print(f"MLP3 (3 clases) -> acc_train: {acc_train:.2f}% | acc_test: {acc_test:.2f}%")


Epoch [100/1000], Loss: 0.1703
Epoch [200/1000], Loss: 0.0222
Epoch [300/1000], Loss: 0.0097
Epoch [400/1000], Loss: 0.0057
Epoch [500/1000], Loss: 0.0039
Epoch [600/1000], Loss: 0.0029
Epoch [700/1000], Loss: 0.0022
Epoch [800/1000], Loss: 0.0018
Epoch [900/1000], Loss: 0.0015
Epoch [1000/1000], Loss: 0.0013
MLP3 (3 clases) -> acc_train: 100.00% | acc_test: 97.78%
